In [ ]:
import torch
from torch.utils.data import DataLoader
from torchvision import transforms, datasets
from former import HybridVisionFormer
import matplotlib.pyplot as plt
import os
from tqdm import tqdm
import random

def denormalize(img):
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3,1,1)
    std = torch.tensor([0.229, 0.224, 0.225]).view(3,1,1)
    img = img * std + mean
    return torch.clamp(img, 0, 1)

def show_samples():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    TEST_ROOT = '../dataset/test'
    MODEL_PATH = 'best_model.pth'

    test_transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406],
                             [0.229, 0.224, 0.225])
    ])

    test_dataset = datasets.ImageFolder(root=TEST_ROOT, transform=test_transform)
    test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

    class_names = ["Surprise", "Fear", "Disgust", "Happiness",
                   "Sadness", "Anger", "Neutral"]

    # Load model
    model = HybridVisionFormer(num_classes=7).to(device)

    if not os.path.exists(MODEL_PATH):
        print(f"Không tìm thấy {MODEL_PATH}")
        return

    state_dict = torch.load(MODEL_PATH, map_location=device)
    new_state_dict = {k.replace('module.', ''): v for k, v in state_dict.items()}
    model.load_state_dict(new_state_dict)
    model.eval()

    correct_samples = []
    wrong_samples = []

    with torch.no_grad():
        for imgs, labels in tqdm(test_loader, desc="Scanning test set"):
            imgs = imgs.to(device)

            with torch.amp.autocast('cuda'):
                outputs = model(imgs)

            preds = torch.argmax(outputs, dim=1)

            for i in range(len(imgs)):
                sample = (
                    imgs[i].cpu(),
                    labels[i].item(),
                    preds[i].item()
                )

                if labels[i] == preds[i]:
                    correct_samples.append(sample)
                else:
                    wrong_samples.append(sample)

    # Random chọn 3
    correct_show = random.sample(correct_samples, min(3, len(correct_samples)))
    wrong_show = random.sample(wrong_samples, min(3, len(wrong_samples)))

    # ===== SHOW CORRECT =====
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    fig.suptitle("Correct Predictions", fontsize=16)

    for ax, (img, real, pred) in zip(axes, correct_show):
        img = denormalize(img).permute(1,2,0).numpy()
        ax.imshow(img)
        ax.set_title(f"Real: {class_names[real]}\nPred: {class_names[pred]}",
                     color='green')
        ax.axis('off')

    plt.tight_layout()
    plt.show()

    # ===== SHOW WRONG =====
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    fig.suptitle("Wrong Predictions", fontsize=16)

    for ax, (img, real, pred) in zip(axes, wrong_show):
        img = denormalize(img).permute(1,2,0).numpy()
        ax.imshow(img)
        ax.set_title(f"Real: {class_names[real]}\nPred: {class_names[pred]}",
                     color='red')
        ax.axis('off')

    plt.tight_layout()
    plt.show()


if __name__ == "__main__":
    show_samples()